In [ ]:
!pip install -q tensorflow datasets sentencepiece sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.1 MB/s eta 0:00:00


In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np

from datasets import load_dataset

from sklearn.model_selection import train_test_split

print(tf.__version__)

2.20.0


In [ ]:
dataset = load_dataset("cfilt/iitb-english-hindi")

README.md:   0%|          | 0.00/3.14k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/190M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/85.7k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/500k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1659083 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/520 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2507 [00:00<?, ? examples/s]

In [ ]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 1659083
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 520
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 2507
    })
})


In [ ]:
train_df = dataset["train"].to_pandas()

In [ ]:
train_df.head()

,translation
0,{'en': 'Give your application an accessibility...
1,"{'en': 'Accerciser Accessibility Explorer', 'h..."
2,{'en': 'The default plugin layout for the bott...
3,{'en': 'The default plugin layout for the top ...
4,{'en': 'A list of plugins that are disabled by...


In [ ]:
train_df["english"] = train_df["translation"].apply(lambda x: x["en"])

train_df["hindi"] = train_df["translation"].apply(lambda x: x["hi"])

In [ ]:
train_df = train_df[["english","hindi"]]

In [ ]:
train_df.head()

,english,hindi
0,Give your application an accessibility workout,अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें
1,Accerciser Accessibility Explorer,एक्सेर्साइसर पहुंचनीयता अन्वेषक
2,The default plugin layout for the bottom panel,निचले पटल के लिए डिफोल्ट प्लग-इन खाका
3,The default plugin layout for the top panel,ऊपरी पटल के लिए डिफोल्ट प्लग-इन खाका
4,A list of plugins that are disabled by default,उन प्लग-इनों की सूची जिन्हें डिफोल्ट रूप से नि...


In [ ]:
print(train_df.shape)

(1659083, 2)


In [ ]:
# Increasing the dataset size to 50,000 for better vocabulary coverage
train_df = train_df.iloc[:50000]

In [ ]:
print(train_df.isnull().sum())
train_df = train_df.dropna()

english    0
hindi      0
dtype: int64


In [ ]:
train_df = train_df.drop_duplicates()

In [ ]:
train_df = train_df.reset_index(drop=True)

In [ ]:
print(train_df.head())

print(train_df.shape)

                                          english  \
0  Give your application an accessibility workout   
1               Accerciser Accessibility Explorer   
2  The default plugin layout for the bottom panel   
3     The default plugin layout for the top panel   
4  A list of plugins that are disabled by default   

                                               hindi  
0    अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें  
1                    एक्सेर्साइसर पहुंचनीयता अन्वेषक  
2              निचले पटल के लिए डिफोल्ट प्लग-इन खाका  
3               ऊपरी पटल के लिए डिफोल्ट प्लग-इन खाका  
4  उन प्लग-इनों की सूची जिन्हें डिफोल्ट रूप से नि...  
(10673, 2)


In [ ]:
train_df["hindi"] = (
    "<start> "
    + train_df["hindi"]
    + " <end>"
)

In [ ]:
train_df.head()

,english,hindi
0,Give your application an accessibility workout,<start> अपने अनुप्रयोग को पहुंचनीयता व्यायाम क...
1,Accerciser Accessibility Explorer,<start> एक्सेर्साइसर पहुंचनीयता अन्वेषक <end>
2,The default plugin layout for the bottom panel,<start> निचले पटल के लिए डिफोल्ट प्लग-इन खाका ...
3,The default plugin layout for the top panel,<start> ऊपरी पटल के लिए डिफोल्ट प्लग-इन खाका <...
4,A list of plugins that are disabled by default,<start> उन प्लग-इनों की सूची जिन्हें डिफोल्ट र...


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

# Re-fitting tokenizer on the larger 50k dataset
english_tokenizer = Tokenizer(filters="")
english_tokenizer.fit_on_texts(train_df["english"])

hindi_tokenizer = Tokenizer(filters="")
hindi_tokenizer.fit_on_texts(train_df["hindi"])

# Updating vocab sizes
VOCAB_SIZE_EN = len(english_tokenizer.word_index) + 1
VOCAB_SIZE_HI = len(hindi_tokenizer.word_index) + 1
print(f"New English Vocab: {VOCAB_SIZE_EN}")
print(f"New Hindi Vocab: {VOCAB_SIZE_HI}")

In [ ]:
hindi_tokenizer = Tokenizer(
    filters=""
)

hindi_tokenizer.fit_on_texts(
    train_df["hindi"]
)

In [ ]:
english_vocab = len(
    english_tokenizer.word_index
) + 1

hindi_vocab = len(
    hindi_tokenizer.word_index
) + 1

print(english_vocab)
print(hindi_vocab)

5495
6027


In [ ]:
encoder_sequences = english_tokenizer.texts_to_sequences(
    train_df["english"]
)

In [ ]:
decoder_sequences = hindi_tokenizer.texts_to_sequences(
    train_df["hindi"]
)

In [ ]:
print(train_df["english"][0])
print(encoder_sequences[0])

print()

print(train_df["hindi"][0])
print(decoder_sequences[0])

Give your application an accessibility workout
[1767, 51, 236, 23, 597, 3163]

<start> अपने अनुप्रयोग को पहुंचनीयता व्यायाम का लाभ दें <end>
[1, 393, 164, 4, 1233, 3418, 7, 3419, 132, 2]


In [ ]:
max_encoder_len = max(
    len(sentence)
    for sentence in encoder_sequences
)

max_decoder_len = max(
    len(sentence)
    for sentence in decoder_sequences
)

print(max_encoder_len)
print(max_decoder_len)

117
116


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Generating new sequences for the larger dataset
encoder_sequences = english_tokenizer.texts_to_sequences(train_df["english"])
decoder_sequences = hindi_tokenizer.texts_to_sequences(train_df["hindi"])

encoder_input = pad_sequences(encoder_sequences, maxlen=MAX_LEN_EN, padding="post")
decoder_input = pad_sequences(decoder_sequences, maxlen=MAX_LEN_HI, padding="post")

decoder_target = []
for sentence in decoder_input:
    decoder_target.append(sentence[1:])

decoder_target = pad_sequences(decoder_target, maxlen=MAX_LEN_HI-1, padding="post")
print("Sequences updated for 50k rows.")

In [ ]:
print(encoder_input.shape)

print(decoder_input.shape)

(10673, 117)
(10673, 116)


In [ ]:
decoder_target = []

for sentence in decoder_input:

    decoder_target.append(

        sentence[1:]

    )

decoder_target = pad_sequences(

    decoder_target,

    maxlen=max_decoder_len-1,

    padding="post"

)

In [ ]:
VOCAB_SIZE_EN = english_vocab
VOCAB_SIZE_HI = hindi_vocab

MAX_LEN_EN = max_encoder_len
MAX_LEN_HI = max_decoder_len

D_MODEL = 256

NUM_HEADS = 8

DFF = 512

DROPOUT = 0.1

In [ ]:
encoder_embedding = tf.keras.layers.Embedding(
    input_dim=VOCAB_SIZE_EN,
    output_dim=D_MODEL
)

In [ ]:
decoder_embedding = tf.keras.layers.Embedding(
    input_dim=VOCAB_SIZE_HI,
    output_dim=D_MODEL
)

In [ ]:
def positional_encoding(length, depth):
    depth = depth / 2

    positions = np.arange(length)[:, np.newaxis]
    depths = np.arange(depth)[np.newaxis, :] / depth

    angle_rates = 1 / (10000**depths)
    angle_rads = positions * angle_rates

    pos_encoding = np.concatenate(
        [np.sin(angle_rads), np.cos(angle_rads)],
        axis=-1
    )

    return tf.cast(pos_encoding, dtype=tf.float32)

encoder_pos = positional_encoding(
    MAX_LEN_EN,
    D_MODEL
)

decoder_pos = positional_encoding(
    MAX_LEN_HI,
    D_MODEL
)

In [ ]:
encoder_output = (
    encoder_embedding(encoder_input)
    + encoder_pos
)


encoder_output = (
    encoder_embedding(encoder_input)
    + encoder_pos
)

In [ ]:
attention = tf.keras.layers.MultiHeadAttention(
    num_heads=NUM_HEADS,
    key_dim=D_MODEL // NUM_HEADS
)

In [ ]:
encoder_output = attention(

    query=encoder_output[:100],

    value=encoder_output[:100],

    key=encoder_output[:100]

)

In [ ]:
encoder_inputs = tf.keras.Input(
    shape=(MAX_LEN_EN,),
    name="encoder_input"
)

In [ ]:
decoder_inputs = tf.keras.Input(
    shape=(MAX_LEN_HI-1,),
    name="decoder_input"
)

In [ ]:
enc_embed = tf.keras.layers.Embedding(
    VOCAB_SIZE_EN,
    D_MODEL
)(encoder_inputs)

In [ ]:
dec_embed = tf.keras.layers.Embedding(
    VOCAB_SIZE_HI,
    D_MODEL
)(decoder_inputs)

In [ ]:
enc_embed = enc_embed + encoder_pos

dec_embed = dec_embed + decoder_pos[:-1]

In [ ]:
encoder_attention = tf.keras.layers.MultiHeadAttention(

    num_heads=NUM_HEADS,

    key_dim=D_MODEL//NUM_HEADS

)

In [ ]:
encoder_output = encoder_attention(

    query=enc_embed,

    key=enc_embed,

    value=enc_embed
)

In [ ]:
decoder_attention = tf.keras.layers.MultiHeadAttention(

    num_heads=NUM_HEADS,

    key_dim=D_MODEL//NUM_HEADS
)

In [ ]:
decoder_output = decoder_attention(

    query=dec_embed,

    key=dec_embed,

    value=dec_embed
)

In [ ]:
cross_attention = tf.keras.layers.MultiHeadAttention(

    num_heads=NUM_HEADS,

    key_dim=D_MODEL//NUM_HEADS
)

In [ ]:
decoder_output = cross_attention(

    query=decoder_output,

    key=encoder_output,

    value=encoder_output
)

In [ ]:
# Adding missing Residual Connections and Normalization for Encoder and Decoder
# Encoder Step improvement
x_enc = tf.keras.layers.Add()([enc_embed, encoder_output])
x_enc = tf.keras.layers.LayerNormalization()(x_enc)

# Decoder Step improvement
x_dec = tf.keras.layers.Add()([dec_embed, decoder_output])
x_dec = tf.keras.layers.LayerNormalization()(x_dec)

# Feed Forward Network
ffn = tf.keras.Sequential([
    tf.keras.layers.Dense(DFF, activation="relu"),
    tf.keras.layers.Dense(D_MODEL)
])

x = tf.keras.layers.Add()([x_dec, ffn(x_dec)])
x = tf.keras.layers.LayerNormalization()(x)

outputs = tf.keras.layers.Dense(VOCAB_SIZE_HI, activation="softmax")(x)

In [ ]:
model = tf.keras.Model(

    inputs=[

        encoder_inputs,

        decoder_inputs

    ],

    outputs=outputs
)

In [ ]:
model.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

In [ ]:
# Re-running training with the larger dataset and higher epochs
history = model.fit(
    [
        encoder_input,
        decoder_input[:,:-1]
    ],
    decoder_target,
    epochs=30,
    batch_size=128,
    validation_split=0.1,
    callbacks=[checkpoint, early]
)

Epoch 1/30
76/76 ━━━━━━━━━━━━━━━━━━━━ 23s 211ms/step - accuracy: 0.9591 - loss: 0.2566 - val_accuracy: 0.9446 - val_loss: 0.4773
Epoch 2/30
76/76 ━━━━━━━━━━━━━━━━━━━━ 9s 112ms/step - accuracy: 0.9593 - loss: 0.2516 - val_accuracy: 0.9447 - val_loss: 0.4885
Epoch 3/30
76/76 ━━━━━━━━━━━━━━━━━━━━ 8s 111ms/step - accuracy: 0.9594 - loss: 0.2487 - val_accuracy: 0.9449 - val_loss: 0.4941


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
checkpoint = tf.keras.callbacks.ModelCheckpoint(

    "best_transformer.keras",

    monitor="val_loss",

    save_best_only=True

)

In [ ]:
early = tf.keras.callbacks.EarlyStopping(

    monitor="val_loss",

    patience=3,

    restore_best_weights=True

)

In [ ]:
history = model.fit(

    [

        encoder_input,

        decoder_input[:,:-1]

    ],

    decoder_target,

    validation_split=0.2,

    batch_size=64,

    epochs=20,

    callbacks=[checkpoint,early]

)

Epoch 1/20
134/134 ━━━━━━━━━━━━━━━━━━━━ 24s 122ms/step - accuracy: 0.9481 - loss: 0.4642 - val_accuracy: 0.9416 - val_loss: 0.6612
Epoch 2/20
134/134 ━━━━━━━━━━━━━━━━━━━━ 8s 60ms/step - accuracy: 0.9481 - loss: 0.4604 - val_accuracy: 0.9416 - val_loss: 0.6630
Epoch 3/20
134/134 ━━━━━━━━━━━━━━━━━━━━ 8s 60ms/step - accuracy: 0.9481 - loss: 0.4590 - val_accuracy: 0.9416 - val_loss: 0.6644
Epoch 4/20
134/134 ━━━━━━━━━━━━━━━━━━━━ 8s 61ms/step - accuracy: 0.9481 - loss: 0.4571 - val_accuracy: 0.9416 - val_loss: 0.6625


In [ ]:
model.save("translator.keras")

In [ ]:
model = tf.keras.models.load_model(

    "translator.keras"

)

In [ ]:
sequence = english_tokenizer.texts_to_sequences(

    ["How are you?"]

)

In [ ]:
sequence = pad_sequences(

    sequence,

    maxlen=max_encoder_len,

    padding="post"

)

In [ ]:
def beam_search(model, sequence, beam_width=3, min_length=2):
    start_token = hindi_tokenizer.word_index['<start>']
    end_token = hindi_tokenizer.word_index['<end>']

    beams = [([start_token], 0.0)]

    for i in range(MAX_LEN_HI - 1):
        candidates = []
        for seq, score in beams:
            if seq[-1] == end_token:
                candidates.append((seq, score))
                continue

            temp_input = np.zeros((1, MAX_LEN_HI - 1))
            temp_input[0, :len(seq)] = seq

            prediction = model.predict([sequence, temp_input], verbose=0)
            probs = prediction[0, len(seq)-1, :]

            # Penalize end_token if sequence is too short
            if i < min_length:
                probs[end_token] = 0.0
                probs = probs / np.sum(probs)

            top_k = np.argsort(probs)[-beam_width:]
            for idx in top_k:
                candidates.append((seq + [idx], score + np.log(probs[idx] + 1e-10)))

        beams = sorted(candidates, key=lambda x: x[1], reverse=True)[:beam_width]
        if all(s[-1] == end_token for s, _ in beams): break

    return beams[0][0]

# Re-run prediction
best_seq = beam_search(model, sequence)
final_text = [reverse_index.get(token, "") for token in best_seq if token not in [0, 1, 2]]
print("Input English: How are you?")
print("Predicted Hindi (Beam Search):", " ".join(final_text))

Input English: How are you?
Predicted Hindi (Beam Search): करें


In [ ]:
reverse_index = {

v:k

for k,v in hindi_tokenizer.word_index.items()

}

In [ ]:
translated = []

for token in decoder_sequence:

    translated.append(

        reverse_index.get(token,"")

    )

print(

" ".join(translated)

)

<start> करें
